# 05 - Neuron bridge | Step 1: characterise the neuroblast -> neuron transition

**Why this assay exists.** Figure 1 of the paper analyses APA (alternative polyadenylation / 3'UTR
isoform usage) along the adult-SVZ neurogenic pseudotime, but that trajectory **stops at neuroblasts
(NB)**. Figure 2 (human cortical organoids) runs all the way into *maturing neurons*. To strengthen
the Fig 1 -> Fig 2 bridge we ask a focused question on the mouse side: **does the 3'UTR-length
programme continue, plateau, or reverse as neuroblasts become neurons?**

**Why a *separate, discrete* assay (not an extension of the main pseudotime).** The `Neurons` cluster
is a distinct terminal population, not a smooth continuation of the NB trajectory. This first notebook
quantifies that, so the discrete design is defensible to a reviewer. We establish:

1. **Section A** - the neuron population and its (shallow) sequencing depth, per batch.
2. **Section B** - the evidence that neurons are a *discrete* state: they are essentially unplaceable
   by the lineage diffusion-pseudotime, they sit apart on the UMAP, and their nearest-neighbour graph
   connectivity is dominated by other neurons (with neuroblasts as the only meaningful external link).
3. **Section C** - the per-gene coverage budget, which dictates that the **per-gene** Delta-psi arm is
   limited to a few hundred adequately-covered genes, so the **global per-cell W shift** (Step 3) is
   the robust headline and per-gene Delta-psi (Steps 4-5) is an exploratory, coverage-gated layer.

All effect sizes in this assay are reported **without per-gene p-values** (n = 209 neurons in 2
batches): Delta-psi and W are bounded effect sizes, consistent with the paper's defensibility rules.

**Outputs:** `DirMult_neuron_bridge/neuron_jump_characterisation.csv` (per-gene coverage in each group)
and the figures under `DirMult_neuron_bridge/figures/`.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "downstr.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/Wildtype_mice/2026/05-neuron_bridge")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
%matplotlib inline
import os
import numpy as np
import pandas as pd
import anndata as ad
import scipy.sparse as sp
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 150, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

BASE = f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling"
APA  = f"{BASE}/UTR_count_table_genelevel_bw12_above10cts_apa_named.h5ad"   # raw PAS counts (all cells, incl. neurons)
HARM = f"{BASE}/UTR_count_table_genelevel_bw12_above10cts_apa_named_adata_harmony.h5ad"  # same cells + UMAP + kNN graph
OUT  = f"{BASE}/DirMult_neuron_bridge"
FIG  = f"{OUT}/figures"
os.makedirs(FIG, exist_ok=True)

# Lineage order (stem -> committed) plus the terminal Neurons; fixed colours for every notebook in this assay
LINE = ['qNSC1', 'qNSC2', 'aNSC1', 'aNSC2', 'TAP', 'NB', 'Neurons']
COL  = dict(zip(LINE, ['#4575b4', '#74add1', '#abd9e9', '#fee090', '#fdae61', '#f46d43', '#d73027']))
BATCHES = ['3', '4']   # the only batches with a meaningful neuron count (118 + 91 = 209 of 214)
print("paths and style set; neuron-containing batches =", BATCHES)

## Section A - The neuron population and its sequencing depth

The `Neurons` label marks 214 cells. Two facts that shape the whole assay:

- **Batch concentration.** Neurons are essentially confined to batches 3 and 4 (118 + 91 = 209 of 214).
  We therefore run every contrast **within batches 3+4** and weight the two batches equally, so the
  NB-vs-neuron comparison is not confounded by batch composition.
- **Shallow depth.** This is the PAS-site count matrix (only reads at poly(A) sites), so per-cell depth
  is low (~100-150 UMIs/cell). That is the root cause of the per-gene coverage limit in Section C.

The left plot shows neurons per batch; the right shows median UMIs/cell per cell type (batches 3+4) -
neurons are no shallower than neuroblasts, so the limit is cell *number*, not per-cell quality.

In [ ]:
a = ad.read_h5ad(APA)
ct = a.obs['celltype'].astype(str)
bt = a.obs['batch_number'].astype(str)
X  = a.X.tocsr() if sp.issparse(a.X) else sp.csr_matrix(a.X)
inb = bt.isin(BATCHES).values

xt = pd.crosstab(ct, bt)
print("Total Neurons:", int((ct == 'Neurons').sum()), " | in batches 3+4:", int(((ct=='Neurons').values & inb).sum()))
print("\nLineage cell-type x batch:")
display(xt.loc[LINE])

depth = []
for l in LINE:
    m = (ct == l).values & inb
    rs = np.asarray(X[m].sum(1)).ravel()
    depth.append({'celltype': l, 'n_cells_b34': int(m.sum()),
                  'median_UMI_per_cell': float(np.median(rs)) if m.sum() else np.nan})
depth = pd.DataFrame(depth)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
npb = xt.loc['Neurons']
ax[0].bar(npb.index.astype(str), npb.values, color=COL['Neurons'])
ax[0].set(title='Neurons per batch (assay uses 3+4)', xlabel='batch', ylabel='neuron cells')
ax[1].bar(depth['celltype'], depth['median_UMI_per_cell'], color=[COL[l] for l in depth['celltype']])
ax[1].set(title='Median UMIs / cell (batches 3+4)', ylabel='median UMI/cell')
ax[1].tick_params(axis='x', rotation=45)
fig.tight_layout(); fig.savefig(f"{FIG}/s1A_neuron_counts_depth.png", bbox_inches='tight'); plt.show()
display(depth)

## Section B - Neurons are a *discrete* terminal state, not a pseudotime continuation

Three independent lines of evidence, all pointing the same way:

1. **The diffusion-pseudotime cannot place them.** In the file the Delta-psi pipeline reads, only
   **1 of 214** neurons received a (non-NaN) `dpt_pseudotime` value - the lineage DPT, computed on the
   continuous qNSC->NB manifold, leaves 213/214 neurons unplaced. (This is also why selecting neurons
   by *celltype*, never by pseudotime, is essential - selecting on pseudotime would silently shrink the
   neuron group to n = 1.)
2. **They sit apart on the UMAP.** The NB->Neuron centroid step is the largest in the lineage and is
   ~1.2x the NB->TAP step.
3. **Their graph neighbourhood is self-contained.** Averaged over neurons, the great majority of a
   neuron's kNN-graph connectivity goes to other neurons; of the remainder, **neuroblasts are the only
   meaningful external partner** (every non-neuronal population other than NB is <1%). So neurons are a
   distinct community whose sole developmental bridge is to NB - exactly the picture that motivates a
   discrete NB->neuron contrast rather than a spline through a gap.

In [ ]:
# 1. DPT coverage per cell type (neurons are essentially excluded from the lineage pseudotime)
pt = pd.to_numeric(a.obs['dpt_pseudotime'], errors='coerce')
covpt = pd.DataFrame({'ct': ct.values, 'has_pt': pt.notna().values}).groupby('ct')['has_pt'].agg(['sum', 'count'])
covpt['frac_with_pseudotime'] = covpt['sum'] / covpt['count']
print("Diffusion-pseudotime coverage by cell type (note Neurons):")
display(covpt.loc[LINE].rename(columns={'sum': 'n_with_pt', 'count': 'n_cells'}))

# 2-3. UMAP + kNN graph from the harmony object (backed: we only touch obs / obsm / obsp, never X)
h   = ad.read_h5ad(HARM, backed='r')
hct = h.obs['celltype'].astype(str).values
um  = np.asarray(h.obsm['X_umap'])

fig, ax = plt.subplots(figsize=(6.2, 5.2))
other = ~np.isin(hct, LINE)
ax.scatter(um[other, 0], um[other, 1], s=3, c='lightgrey', linewidths=0, label='other')
for l in LINE:
    m = hct == l
    ax.scatter(um[m, 0], um[m, 1], s=(16 if l == 'Neurons' else 5), c=COL[l], linewidths=0, label=l)
ax.set(title='UMAP: neurogenic lineage + terminal Neurons', xticks=[], yticks=[])
ax.legend(markerscale=2, fontsize=7, loc='best', frameon=False)
fig.savefig(f"{FIG}/s1B_umap_lineage_neurons.png", bbox_inches='tight'); plt.show()

cent = {l: um[hct == l].mean(0) for l in LINE}
dist = lambda x, y: float(np.linalg.norm(cent[x] - cent[y]))
steps = [('qNSC1','qNSC2'),('qNSC2','aNSC1'),('aNSC1','aNSC2'),('aNSC2','TAP'),('TAP','NB'),('NB','Neurons')]
dd = pd.DataFrame([{'step': f'{x}->{y}', 'umap_dist': dist(x, y)} for x, y in steps])
ratio = dist('NB', 'Neurons') / dist('NB', 'TAP')

C  = h.obsp['connectivities'].tocsr()
neu = np.where(hct == 'Neurons')[0]
Cn = C[neu]; tot = np.asarray(Cn.sum(1)).ravel(); tot[tot == 0] = np.nan
grp = {'Neurons': hct == 'Neurons', 'NB': hct == 'NB', 'TAP': hct == 'TAP',
       'other NSC': np.isin(hct, ['qNSC1', 'qNSC2', 'aNSC1', 'aNSC2']), 'glia/other': ~np.isin(hct, LINE)}
fr = {g: float(np.nanmean(np.asarray(Cn[:, m].sum(1)).ravel() / tot) * 100) for g, m in grp.items()}

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
ax[0].bar(dd['step'], dd['umap_dist'], color=[COL[s.split('->')[1]] for s in dd['step']])
ax[0].set(title=f'UMAP step distances  (NB->Neuron = {ratio:.2f}x NB->TAP)', ylabel='UMAP distance')
ax[0].tick_params(axis='x', rotation=45)
ax[1].bar(list(fr.keys()), list(fr.values()), color=['#d73027', '#f46d43', '#fdae61', '#74add1', 'lightgrey'])
ax[1].set(title="Where a neuron's graph connectivity goes", ylabel='% of connectivity')
ax[1].tick_params(axis='x', rotation=30)
fig.tight_layout(); fig.savefig(f"{FIG}/s1B_jump_metrics.png", bbox_inches='tight'); plt.show()
print("Neuron kNN-connectivity fractions (%):", {k: round(v, 1) for k, v in fr.items()})
print(f"NB<->Neuron / NB<->TAP UMAP ratio = {ratio:.2f}")

## Section C - Per-gene coverage budget: why global W is the headline

For every multi-site gene we count, **within the neuron group (batches 3+4)**, how many cells express
it and how many UMIs it has - and likewise for the NB and qNSC comparison groups. Because the matrix is
shallow and neurons are few, the median gene is seen in only ~1 neuron; only a few hundred genes clear a
usable coverage gate in *both* groups.

Consequence, baked into the assay design:

- The **per-gene Delta-psi** contrast (Steps 2/4/5) is computed at a permissive floor (>=10 cells AND
  >=20 UMIs per group -> 234 NB->neuron genes) with the per-gene coverage stored, so downstream notebooks
  can tighten the gate and report sensitivity. It is an **exploratory, coverage-gated** layer.
- The **global per-cell W shift** (Step 3) averages over *all* genes a cell expresses, so it is immune
  to single-gene sparsity. It is the **robust headline** for "do neuron 3'UTRs differ from neuroblast
  3'UTRs overall?".

This notebook writes the per-gene coverage table that all downstream steps use to gate genes.

In [ ]:
genes = a.var['utr_name'].astype(str).values
uniq, inv = np.unique(genes, return_inverse=True)
nfeat = np.bincount(inv)
multi = nfeat >= 2
G = sp.csr_matrix((np.ones(len(inv)), (np.arange(len(inv)), inv)), shape=(len(inv), len(uniq)))

def cov(mask):
    gc = X[mask] @ G                      # (cells x genes) gene-level counts
    return np.asarray((gc > 0).sum(0)).ravel(), np.asarray(gc.sum(0)).ravel()

cN, uN = cov((ct == 'Neurons').values & inb)
cB, uB = cov((ct == 'NB').values & inb)
cQ, uQ = cov(np.isin(ct, ['qNSC1', 'qNSC2']) & inb)
covg = pd.DataFrame({'gene_id': uniq, 'multi': multi,
                     'cells_neuron': cN, 'umi_neuron': uN, 'cells_NB': cB, 'umi_NB': uB,
                     'cells_qNSC': cQ, 'umi_qNSC': uQ})
covg = covg[covg['multi']].drop(columns='multi').copy()
g2n = a.var.drop_duplicates('utr_name').set_index('utr_name')['gene_name']
covg['gene_name'] = covg['gene_id'].map(g2n)
covg.to_csv(f"{OUT}/neuron_jump_characterisation.csv", index=False)
print(f"wrote neuron_jump_characterisation.csv  ({len(covg)} multi-site genes)")

rows = []
for mc, mu in [(30, 100), (20, 50), (15, 40), (10, 30), (10, 20)]:
    okN = int(((covg.cells_neuron >= mc) & (covg.umi_neuron >= mu) &
               (covg.cells_NB >= mc) & (covg.umi_NB >= mu)).sum())
    okQ = int(((covg.cells_neuron >= mc) & (covg.umi_neuron >= mu) &
               (covg.cells_qNSC >= mc) & (covg.umi_qNSC >= mu)).sum())
    rows.append({'min_cells': mc, 'min_umi': mu, 'NB_to_neuron_genes': okN, 'qNSC_to_neuron_genes': okQ})
thr = pd.DataFrame(rows)
print("Testable genes vs coverage gate:")
display(thr)

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
ax[0].hist(np.clip(covg.cells_neuron, 0, 60), bins=60, color=COL['Neurons'])
ax[0].axvline(10, c='k', ls='--', lw=1)
ax[0].set(title='Expressing neuron cells per gene', xlabel='cells (clipped at 60)', ylabel='genes', yscale='log')
lab = (thr.min_cells.astype(str) + 'c/' + thr.min_umi.astype(str) + 'u')
ax[1].plot(lab, thr.NB_to_neuron_genes, 'o-', color='#333', label='NB->neuron')
ax[1].plot(lab, thr.qNSC_to_neuron_genes, 's--', color='#888', label='qNSC->neuron')
ax[1].set(title='Testable genes vs gate', xlabel='gate (min cells / min UMIs)', ylabel='genes')
ax[1].tick_params(axis='x', rotation=30); ax[1].legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(f"{FIG}/s1C_coverage_qc.png", bbox_inches='tight'); plt.show()
print("Chosen floor for the per-gene arm: 10 cells / 20 UMIs per group (-> 234 NB->neuron genes).")
print("Per-gene Delta-psi is EXPLORATORY/coverage-gated; the global per-cell W shift (Step 3) is the robust headline.")

## Takeaways

- Neurons are a **discrete terminal state**: unplaceable by the lineage pseudotime (1/214), spatially
  separated on the UMAP, and a self-contained graph community whose only meaningful external link is to
  neuroblasts. A **discrete NB->neuron contrast** is therefore the statistically honest design.
- The contrast must be run **within batches 3+4** (where the neurons are) with **equal batch weight**,
  and neurons must be selected by **cell type**, never by pseudotime.
- Per-gene resolution is **coverage-limited** (~234 genes at the chosen floor), so the per-gene
  Delta-psi / cross-species results are framed as exploratory; the **global per-cell W shift** is the
  robust, all-gene headline.

Next: **Step 2** (already run) produced `delta_psi_nb_to_neuron_per_gene.csv` and
`delta_psi_qnsc_to_neuron_per_gene.csv`. **Step 3** computes the global per-cell W shift.